# Pipeline Development Notebook

Calls the production functions in `server/core/` directly — no local function
definitions.  Use this to:

1. **Run the pipeline** with the mock scenario (or your own streams).
2. **Inspect every intermediate** DataFrame.
3. **Sensecheck** via validation + interactive Plotly charts.
4. **Export** pipeline snapshots for the LLM test CLI.

If you change `server/core/`, restart the kernel and re-run to pick up changes.

In [20]:
from __future__ import annotations

import datetime as dt
import json
import sys
from pathlib import Path

import plotly.graph_objects as go
import polars as pl
from plotly.subplots import make_subplots

# Ensure the project root is on sys.path so server.core is importable
_project_root = str(Path("..").resolve())
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

In [ ]:
# ── Imports from server.core ─────────────────────────────────────────────
from server.core.config import BlockConfig, StreamConfig, SECONDS_PER_YEAR
from server.core.helpers import annualize, deannualize, raw_to_target_expr
from server.core.pipeline import (
    build_blocks_df,
    attach_market_values,
    build_time_grid,
    run_pipeline,
)
from server.core.transforms import get_registry, get_step, from_dict
from server.core.serializers import snapshot_from_pipeline
from server.core.mock_scenario import (
    MOCK_NOW,
    MOCK_BANKROLL,
    MOCK_SMOOTHING_HL_SECS,
    MOCK_TIME_GRID_INTERVAL,
    MOCK_RISK_DIMENSION_COLS,
    MOCK_STREAMS,
    MOCK_MARKET_PRICING,
)

## Transform Registry

The generalized transform system exposes 7 pipeline steps, each with a library
of pluggable implementations.  The cell below inspects the live registry.

In [ ]:
# ── Transform Registry: list all steps, transforms, and param schemas ────
registry = get_registry()

for step_name in registry.list_steps():
    lib = registry.get_step(step_name)
    selected = lib.get_selected()
    print(f"Step: {step_name}")
    print(f"  Contract: {lib.contract_doc}")
    print(f"  Selected: {selected.name}")
    for reg in lib.list_transforms():
        marker = " <-- active" if reg.name == selected.name else ""
        print(f"    [{reg.name}]{marker}: {reg.description}")
        if reg.params:
            for p in reg.params:
                extras = []
                if p.min is not None:
                    extras.append(f"min={p.min}")
                if p.max is not None:
                    extras.append(f"max={p.max}")
                if p.options is not None:
                    extras.append(f"options={p.options}")
                extra_str = f" ({', '.join(extras)})" if extras else ""
                print(f"      - {p.name}: {p.type} = {p.default!r}{extra_str}  {p.description}")
    print()

## Scenario Config

Override any mock defaults here.  To use the mock scenario unchanged, just
run the cell as-is.

In [22]:
# ── Scenario parameters (edit to override mock defaults) ─────────────────
now = MOCK_NOW
bankroll = MOCK_BANKROLL
smoothing_hl_secs = MOCK_SMOOTHING_HL_SECS
time_grid_interval = MOCK_TIME_GRID_INTERVAL
risk_dimension_cols = MOCK_RISK_DIMENSION_COLS

streams = MOCK_STREAMS
market_pricing = MOCK_MARKET_PRICING

### Transform Configuration

Select which transform implementation to use for each pipeline step.
Uncomment alternatives to experiment with different strategies.

In [ ]:
# ── Transform configuration (select implementations per step) ────────────
# Each key is a step name; its value is the transform to use.
# Append "_params" to override that transform's parameters.
# Omitted steps use the registry default (first registered).

transform_config = {
    "unit_conversion":    "affine_power",      # or: "log_scale"
    "decay_profile":      "linear",            # or: "exponential", "sigmoid", "step"
    "temporal_fair_value": "standard",          # or: "flat_forward"
    "variance":           "fair_proportional",  # or: "constant", "squared_fair"
    "aggregation":        "average_offset",     # or: "weighted", "sum_all"
    "position_sizing":    "kelly",              # or: "power_utility"
    "smoothing":          "forward_ewm",        # or: "no_smoothing", "forward_rolling_mean"

    # ── Parameter overrides (uncomment to tweak) ─────────────────────────
    # "decay_profile_params":   {"lam": 5.0},             # exponential decay rate
    # "smoothing_params":       {"half_life_secs": 3600},  # slower EWM smoothing
    # "position_sizing_params": {"risk_aversion": 3.0},    # more conservative sizing
}

print("Transform config:")
for k, v in transform_config.items():
    print(f"  {k}: {v}")

## Run Pipeline

In [ ]:
results = run_pipeline(
    streams=streams,
    market_pricing=market_pricing,
    risk_dimension_cols=risk_dimension_cols,
    now=now,
    bankroll=bankroll,
    smoothing_hl_secs=smoothing_hl_secs,
    time_grid_interval=time_grid_interval,
    transform_config=transform_config,
)

print("=== Block Summary ===")
display(results["blocks_df"].select(
    risk_dimension_cols + [
        "block_name", "stream_name", "space_id", "aggregation_logic",
        "target_value", "target_market_value", "var_fair_ratio",
    ]
))

print("\n=== Time Grid (head) ===")
display(results["time_grid"].head())

print("\n=== Block Fair Values — long format (head) ===")
display(results["block_var_df"].head(10))

print("\n=== Desired Position (head) ===")
display(results["desired_pos_df"].head())

## Compare Transforms

Re-run the pipeline with an alternative transform configuration and compare
the resulting desired positions side-by-side.

In [ ]:
# ── Alternative config: no smoothing + squared_fair variance ─────────────
alt_config = {
    "unit_conversion":    "affine_power",
    "decay_profile":      "linear",
    "temporal_fair_value": "standard",
    "variance":           "squared_fair",       # quadratic variance scaling
    "aggregation":        "average_offset",
    "position_sizing":    "kelly",
    "smoothing":          "no_smoothing",        # raw signal, no EWM
}

alt_results = run_pipeline(
    streams=streams,
    market_pricing=market_pricing,
    risk_dimension_cols=risk_dimension_cols,
    now=now,
    bankroll=bankroll,
    smoothing_hl_secs=smoothing_hl_secs,
    time_grid_interval=time_grid_interval,
    transform_config=alt_config,
)

# ── Compare desired positions ────────────────────────────────────────────
base_pos = results["desired_pos_df"].select(
    risk_dimension_cols + ["timestamp", "smoothed_desired_position"]
).rename({"smoothed_desired_position": "base_pos"})

alt_pos = alt_results["desired_pos_df"].select(
    risk_dimension_cols + ["timestamp", "smoothed_desired_position"]
).rename({"smoothed_desired_position": "alt_pos"})

comparison = base_pos.join(alt_pos, on=risk_dimension_cols + ["timestamp"])
comparison = comparison.with_columns(
    (pl.col("alt_pos") - pl.col("base_pos")).alias("diff"),
    (
        pl.when(pl.col("base_pos").abs() > 1e-9)
        .then((pl.col("alt_pos") - pl.col("base_pos")) / pl.col("base_pos").abs() * 100)
        .otherwise(0.0)
    ).alias("pct_change"),
)

print("=== Transform Comparison: base (defaults) vs alt (no_smoothing + squared_fair) ===")
print(f"  Base config:  smoothing=forward_ewm, variance=fair_proportional")
print(f"  Alt config:   smoothing=no_smoothing, variance=squared_fair")
print()

# Show first and last few rows
print("Head of comparison:")
display(comparison.head(5))
print("\nTail of comparison:")
display(comparison.tail(5))

# Summary statistics
print("\nSummary of position differences:")
display(comparison.select(
    pl.col("diff").abs().mean().alias("mean_abs_diff"),
    pl.col("diff").abs().max().alias("max_abs_diff"),
    pl.col("pct_change").abs().mean().alias("mean_abs_pct_change"),
))

## Validation

In [24]:
# ── Validation summary ───────────────────────────────────────────────────
pos_df = results["desired_pos_df"]
block_var_df = results["block_var_df"]

checks = {
    "total timestamps": pos_df.height,
    "null edge rows": pos_df.filter(pl.col("edge").is_null()).height,
    "null var rows": pos_df.filter(pl.col("var").is_null()).height,
    "zero var rows": pos_df.filter(pl.col("var").abs() < 1e-18).height,
    "null fair blocks": block_var_df.filter(pl.col("fair").is_null()).height,
    "unique blocks": block_var_df["block_name"].n_unique(),
    "unique spaces": block_var_df["space_id"].n_unique(),
    "unique risk dimensions": pos_df.select(risk_dimension_cols).unique().height,
}
print("=== Validation Checks ===")
for k, v in checks.items():
    status = "OK" if ("null" in k or "zero" in k) and v == 0 else ("WARN" if ("null" in k or "zero" in k) and v > 0 else "INFO")
    print(f"  [{status}] {k}: {v}")

=== Validation Checks ===
  [INFO] total timestamps: 1441
  [OK] null edge rows: 0
  [OK] null var rows: 0
  [WARN] zero var rows: 1
  [WARN] null fair blocks: 7
  [INFO] unique blocks: 7
  [INFO] unique spaces: 6
  [INFO] unique risk dimensions: 1


## Pipeline Snapshot Export

Serialize a snapshot at a chosen timestamp — same dict format the LLM
prompts consume.  Useful for verifying the serializer bridge.

In [25]:
SNAPSHOT_TS = dt.datetime(2026, 1, 1, 17, 0)  # <-- change to pick a different moment

snap = snapshot_from_pipeline(
    results=results,
    timestamp=SNAPSHOT_TS,
    risk_dimension_cols=risk_dimension_cols,
    bankroll=bankroll,
    smoothing_hl_secs=smoothing_hl_secs,
    now=now,
)

print(f"=== Pipeline Snapshot at {SNAPSHOT_TS} ===")
print(json.dumps(snap, indent=2, default=str))

=== Pipeline Snapshot at 2026-01-01 17:00:00 ===
{
  "block_summary": [
    {
      "symbol": "BTC",
      "expiry": "2026-01-02 00:00:00",
      "block_name": "rv",
      "stream_name": "rv",
      "space_id": "shifting",
      "aggregation_logic": "average",
      "raw_value": 0.45,
      "target_value": 0.2025,
      "target_market_value": 0.30250000000000005,
      "var_fair_ratio": 1.0,
      "annualized": true,
      "size_type": "fixed",
      "temporal_position": "shifting",
      "decay_end_size_mult": 1.0,
      "decay_rate_prop_per_min": 0.0,
      "fair": 3.850102669404517e-07,
      "market_fair": 5.751387938246256e-07,
      "var": 3.850102669404517e-07
    },
    {
      "symbol": "BTC",
      "expiry": "2026-01-02 00:00:00",
      "block_name": "mean_iv",
      "stream_name": "mean_iv",
      "space_id": "shifting",
      "aggregation_logic": "offset",
      "raw_value": 0.5,
      "target_value": 0.25,
      "target_market_value": 0.30250000000000005,
      "var_fair_r

## Sensechecking

In [26]:
# ── Plotting setup (shared across all graphs below) ──────────────────────
block_var_df = results["block_var_df"]
plot_block = block_var_df.drop_nulls(subset=["fair"])

# Pick first risk dimension for plotting
first_rd = plot_block.select(risk_dimension_cols).unique().sort(risk_dimension_cols).row(0, named=True)
rd_label = " / ".join(f"{k}={v}" for k, v in first_rd.items())
rd_filter = plot_block
for k, v in first_rd.items():
    rd_filter = rd_filter.filter(pl.col(k) == v)

space_ids = sorted(
    rd_filter["space_id"].unique().to_list(),
    key=lambda s: (s != "shifting", s),
)

# Aggregate per (timestamp, space_id) for space-level plots
avg_part = (
    rd_filter.filter(pl.col("aggregation_logic") == "average")
    .group_by("timestamp", "space_id")
    .agg(pl.col("fair").mean().alias("avg_fair"), pl.col("market_fair").mean().alias("avg_mkt"))
)
off_part = (
    rd_filter.filter(pl.col("aggregation_logic") == "offset")
    .group_by("timestamp", "space_id")
    .agg(pl.col("fair").sum().alias("off_fair"), pl.col("market_fair").sum().alias("off_mkt"))
)

space_plot = (
    avg_part.join(off_part, on=["timestamp", "space_id"], how="full", coalesce=True)
    .with_columns(
        pl.col("avg_fair").fill_null(0.0),
        pl.col("off_fair").fill_null(0.0),
        pl.col("avg_mkt").fill_null(0.0),
        pl.col("off_mkt").fill_null(0.0),
    )
    .with_columns(
        (pl.col("avg_fair") + pl.col("off_fair")).alias("total_fair"),
        (pl.col("avg_mkt") + pl.col("off_mkt")).alias("total_mkt"),
    )
    .sort("timestamp")
)

# Desired position filtered to first risk dimension
pos_df = results["desired_pos_df"]
for k, v in first_rd.items():
    pos_df = pos_df.filter(pl.col(k) == v)
pos_df = pos_df.drop_nulls(subset=["raw_desired_position"])
pos_ts = pos_df["timestamp"].to_list()

print(f"Plotting risk dimension: {rd_label}")
print(f"Spaces: {space_ids}")
print(f"Timestamps: {len(pos_ts)}")

Plotting risk dimension: symbol=BTC / expiry=2026-01-02 00:00:00
Spaces: ['shifting', 'static_20260101_000000', 'static_20260101_040000', 'static_20260101_080000', 'static_20260101_120000', 'static_20260101_160000']
Timestamps: 1441


In [27]:
# Graph 1: Fair Value by Space vs Market Implied
fig1 = make_subplots(
    rows=len(space_ids), cols=1, shared_xaxes=True,
    subplot_titles=space_ids, vertical_spacing=0.04,
)

for row_idx, sid in enumerate(space_ids, start=1):
    sp = space_plot.filter(pl.col("space_id") == sid)
    ts = sp["timestamp"].to_list()
    fig1.add_trace(go.Scatter(
        x=ts, y=sp["total_mkt"].to_list(), name="market fair",
        legendgroup="market fair", showlegend=row_idx == 1,
        line=dict(dash="dash", color="rgba(255,255,255,0.6)", width=2),
    ), row=row_idx, col=1)
    fig1.add_trace(go.Scatter(
        x=ts, y=sp["total_fair"].to_list(), name="total fair",
        legendgroup="total fair", showlegend=row_idx == 1,
        line=dict(width=2, color="rgba(0,204,150,1)"),
    ), row=row_idx, col=1)

fig1.update_layout(title=f"Fair Value by Space vs Market Fair — {rd_label}", template="plotly_dark", height=260 * len(space_ids))
fig1.update_xaxes(title_text="Time", row=len(space_ids), col=1)
fig1.update_yaxes(title_text="Fair x dtte")
fig1.show()

In [28]:
# Graph 2: Stacked Variance Contribution by Block
fig2 = go.Figure()

for sid in space_ids:
    sp_blocks = rd_filter.filter(pl.col("space_id") == sid)
    for bn in sp_blocks["block_name"].unique().sort().to_list():
        bdata = sp_blocks.filter(pl.col("block_name") == bn).sort("timestamp")
        fig2.add_trace(go.Scatter(
            x=bdata["timestamp"].to_list(),
            y=bdata["var"].to_list(),
            name=f"{sid} | {bn}",
            stackgroup="one",
            legendgroup=sid,
        ))

fig2.update_layout(
    title=f"Variance Contribution by Block — {rd_label}",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Variance",
)
fig2.show()

In [29]:
# Graph 3: Edge by Space
fig3 = make_subplots(
    rows=len(space_ids), cols=1, shared_xaxes=True,
    subplot_titles=space_ids, vertical_spacing=0.04,
)

for row_idx, sid in enumerate(space_ids, start=1):
    sp = space_plot.filter(pl.col("space_id") == sid)
    edge = (sp["total_fair"] - sp["total_mkt"]).to_list()
    fig3.add_trace(go.Scatter(
        x=sp["timestamp"].to_list(), y=edge,
        name=f"{sid} edge", showlegend=False,
    ), row=row_idx, col=1)

fig3.update_layout(title=f"Edge by Space — {rd_label}", template="plotly_dark", height=260 * len(space_ids))
fig3.update_xaxes(title_text="Time", row=len(space_ids), col=1)
fig3.update_yaxes(title_text="Fair - Market Fair")
fig3.show()

In [30]:
# Graph 4: Smoothed vs Raw — Edge and Variance
fig5 = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=["Edge: Raw vs Smoothed", "Variance: Raw vs Smoothed"],
    vertical_spacing=0.08,
)

fig5.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["edge"].to_list(),
    name="Raw Edge",
    line=dict(width=1, color="rgba(99,110,250,0.4)"),
), row=1, col=1)
fig5.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["smoothed_edge"].to_list(),
    name="Smoothed Edge",
    line=dict(width=2, color="rgba(99,110,250,1)"),
), row=1, col=1)

fig5.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["var"].to_list(),
    name="Raw Variance",
    line=dict(width=1, color="rgba(0,204,150,0.4)"),
), row=2, col=1)
fig5.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["smoothed_var"].to_list(),
    name="Smoothed Variance",
    line=dict(width=2, color="rgba(0,204,150,1)"),
), row=2, col=1)

fig5.update_layout(
    title=f"Smoothed vs Raw Aggregates — {rd_label}",
    template="plotly_dark",
    height=600,
)
fig5.update_xaxes(title_text="Time", row=2, col=1)
fig5.update_yaxes(title_text="Edge", row=1, col=1)
fig5.update_yaxes(title_text="Variance", row=2, col=1)
fig5.show()

In [31]:
# Graph 5: Raw and Smoothed Desired Position
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["raw_desired_position"].to_list(),
    name="Raw Desired Position",
    line=dict(width=1, color="rgba(99,110,250,0.4)"),
))
fig4.add_trace(go.Scatter(
    x=pos_ts, y=pos_df["smoothed_desired_position"].to_list(),
    name="Smoothed Desired Position",
    line=dict(width=2),
))

fig4.update_layout(
    title=f"Desired Position — {rd_label}",
    template="plotly_dark",
    xaxis_title="Time",
    yaxis_title="Desired Position ($)",
)
fig4.show()

## Step-by-Step Pipeline Inspection

Use the cells below to call individual pipeline stages for debugging.
Each stage's output is stored in a separate variable so you can inspect it.

In [ ]:
# ── Setup: apply transform config to registry ───────────────────────────
from_dict(transform_config or {})
unit_fn = get_step("unit_conversion").get_selected()

# ── Stage 1: Build blocks DataFrame ─────────────────────────────────────
blocks_df = build_blocks_df(streams, risk_dimension_cols, unit_fn)
print(f"blocks_df: {blocks_df.shape}")
display(blocks_df)

In [ ]:
# ── Stage 2: Attach market values ───────────────────────────────────────
blocks_df = attach_market_values(blocks_df, market_pricing, unit_fn)
print(f"blocks_df (with market): {blocks_df.shape}")
display(blocks_df.select(
    risk_dimension_cols + [
        "block_name", "stream_name", "space_id",
        "raw_value", "target_value", "market_value", "target_market_value",
    ]
))

In [34]:
# ── Stage 3: Build time grid ────────────────────────────────────────────
time_grid = build_time_grid(blocks_df, risk_dimension_cols, now, interval=time_grid_interval)
print(f"time_grid: {time_grid.shape}")
display(time_grid.head())
display(time_grid.tail())

time_grid: (1441, 4)


timestamp,symbol,expiry,dtte
datetime[μs],str,datetime[μs],f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:01:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:02:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:03:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 00:04:00,"""BTC""",2026-01-02 00:00:00,0.000002


timestamp,symbol,expiry,dtte
datetime[μs],str,datetime[μs],f64
2026-01-01 23:56:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 23:57:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 23:58:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-01 23:59:00,"""BTC""",2026-01-02 00:00:00,0.000002
2026-01-02 00:00:00,"""BTC""",2026-01-02 00:00:00,null


In [ ]:
# ── Stage 4: Compute block fair values ──────────────────────────────────
fair_fn = get_step("temporal_fair_value").get_selected()
decay_fn = get_step("decay_profile").get_selected()
block_fair_df = fair_fn.fn(
    blocks_df, time_grid, risk_dimension_cols, now, decay_fn,
    **get_step("temporal_fair_value").get_param_values(),
)
print(f"block_fair_df: {block_fair_df.shape}")
display(block_fair_df.head(10))

In [ ]:
# ── Stage 5: Compute block variances ────────────────────────────────────
var_fn = get_step("variance").get_selected()
block_var_df = var_fn.fn(block_fair_df, **get_step("variance").get_param_values())
print(f"block_var_df: {block_var_df.shape}")
display(block_var_df.head(10))

In [ ]:
# ── Stage 6: Aggregate by space ─────────────────────────────────────────
agg_fn = get_step("aggregation").get_selected()
space_agg_df = agg_fn.fn(block_var_df, risk_dimension_cols, **get_step("aggregation").get_param_values())
print(f"space_agg_df: {space_agg_df.shape}")
display(space_agg_df.head())
display(space_agg_df.tail())

In [ ]:
# ── Stage 7: Smoothing + position sizing ────────────────────────────────
smooth_fn = get_step("smoothing").get_selected()
smoothed_df = smooth_fn.fn(space_agg_df, risk_dimension_cols, **get_step("smoothing").get_param_values())

pos_fn = get_step("position_sizing").get_selected()
pos_params = get_step("position_sizing").get_param_values()
VAR_FLOOR = 1e-18
desired_pos_df = smoothed_df.with_columns(
    pl.when(pl.col("var").abs() < VAR_FLOOR).then(0.0)
    .otherwise(pos_fn.fn(pl.col("edge"), pl.col("var"), bankroll, **pos_params))
    .alias("raw_desired_position"),
    pl.when(pl.col("smoothed_var").abs() < VAR_FLOOR).then(0.0)
    .otherwise(pos_fn.fn(pl.col("smoothed_edge"), pl.col("smoothed_var"), bankroll, **pos_params))
    .alias("smoothed_desired_position"),
)
print(f"desired_pos_df: {desired_pos_df.shape}")
display(desired_pos_df.head())
display(desired_pos_df.tail())